In [1]:
import pandas as pd
import os
from src.config import QDRANT_URL, QDRANT_API_KEY, OPENAI_API_KEY, OPENAI_MODEL, OPENAI_API_URL,\
     DENSE_EMBEDDING_MODEL_PATH, OPENAI_MODEL_MINI, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, DEVICE, \
     DEEPSEEK_MODEL

from src.generation_metrics import run_generation_evaluation_pipeline, compute_generation_metrics

/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url = os.path.join(PROCESSED_DATA_DIR, "sakila", "sakila_inline_short.sqlite.db")
DATABASE_NAME = "sakila"


sqlite_config = {
    "params": {
        "url": str(url) 
    },
    "type": "sqlite"}

qdrant_config = {"fastembed_model": DENSE_EMBEDDING_MODEL_PATH,
                 "url": QDRANT_URL, 
                 "api_key": QDRANT_API_KEY,
                 "device": DEVICE}

openai_config = {"api_key": OPENAI_API_KEY,
                 "model": OPENAI_MODEL,
                 "base_url": OPENAI_API_URL,
                 "temperature": 0.01}

In [3]:
rewritten_descriptions_csv_path = os.path.join(INTERIM_DATA_DIR, DATABASE_NAME, "query_descriptions", "query_descriptions_inline_short_rewritten.csv")

### Check generation quality using the same SQL-description that was used to generate ground truth SQL-scripts

In [ ]:

predicted_dirs = run_generation_evaluation_pipeline(
    descriptions_csv_path=rewritten_descriptions_csv_path,
    interim_dir=INTERIM_DATA_DIR,
    database_name=DATABASE_NAME,
    models={
        "mini": OPENAI_MODEL_MINI,
        "gpt5": OPENAI_MODEL,
    },
    qdrant_config=qdrant_config,
    openai_config=openai_config,
    comment_styles=("inline", ),
    description_styles=("short", "business"),
    db_config=sqlite_config,
    n_folds=2,
    source_column="query",
)

print("Predicted result dirs:", predicted_dirs)

In [ ]:
ground_truth_results_dir = INTERIM_DATA_DIR / DATABASE_NAME / "results"

generation_metrics_paths_source_column_query = compute_generation_metrics(
    ground_truth_results_dir=ground_truth_results_dir,
    predicted_results_dirs=predicted_dirs,
    descriptions_csv_path=rewritten_descriptions_csv_path,
)

for path in generation_metrics_paths_source_column_query:
    print(path.name)
    df = pd.read_csv(path)
    display(df.groupby("difficulty")['label'].value_counts())

inline_short_mini_query.csv


difficulty  label                               
easy        Everything matches                      2
hard        Not matching number of rows and cols    2
medium      Not generated                           1
            Not matching values                     1
Name: count, dtype: int64

inline_business_mini_query.csv


difficulty  label                               
easy        Not matching values                     2
hard        Not matching number of rows and cols    2
medium      Not generated                           1
            Not matching values                     1
Name: count, dtype: int64

inline_short_gpt5_query.csv


difficulty  label                               
easy        Everything matches                      2
hard        Not matching number of columns          1
            Not matching number of rows and cols    1
medium      Not generated                           1
            Not matching values                     1
Name: count, dtype: int64

inline_business_gpt5_query.csv


difficulty  label                               
easy        Everything matches                      2
hard        Not matching number of columns          1
            Not matching number of rows and cols    1
medium      Not generated                           1
            Not matching values                     1
Name: count, dtype: int64

### Check generation quality using rewritten query descriptions in different styles, instead of using original query description used for generating ground truth SQL-scripts

In [ ]:

predicted_dirs = run_generation_evaluation_pipeline(
    descriptions_csv_path=rewritten_descriptions_csv_path,
    interim_dir=INTERIM_DATA_DIR,
    database_name=DATABASE_NAME,
    models={
        "mini": OPENAI_MODEL_MINI,
        "gpt5": OPENAI_MODEL,
    },
    qdrant_config=qdrant_config,
    openai_config=openai_config,
    comment_styles=("inline", ),
    description_styles=("short", "business",),
    db_config=sqlite_config,
    n_folds=2,
)

print("Predicted result dirs:", predicted_dirs)

In [8]:
ground_truth_results_dir = INTERIM_DATA_DIR / DATABASE_NAME / "results"

generation_metrics_paths = compute_generation_metrics(
    ground_truth_results_dir=ground_truth_results_dir,
    predicted_results_dirs=predicted_dirs,
    descriptions_csv_path=rewritten_descriptions_csv_path,
)

for path in generation_metrics_paths:
    print(path.name)
    df = pd.read_csv(path)
    display(df.groupby("difficulty")['label'].value_counts())

inline_short_mini.csv


difficulty  label                               
easy        Everything matches                      2
hard        Not matching number of rows and cols    2
medium      Not generated                           1
            Not matching values                     1
Name: count, dtype: int64

inline_business_mini.csv


difficulty  label                      
easy        Everything matches             2
hard        Not generated                  1
            Not matching number of rows    1
medium      Not generated                  1
            Not matching number of rows    1
Name: count, dtype: int64

inline_short_gpt5.csv


difficulty  label                               
easy        Everything matches                      2
hard        Not generated                           1
            Not matching number of rows and cols    1
medium      Not generated                           1
            Not matching values                     1
Name: count, dtype: int64

inline_business_gpt5.csv


difficulty  label                               
easy        Everything matches                      1
            Not matching number of columns          1
hard        Not matching number of rows and cols    2
medium      Not generated                           1
            Not matching number of rows and cols    1
Name: count, dtype: int64